# 07 · El pipeline completo y cierre

⏱ **10 min** · Slides: 4 (el proyecto), 33–34 (lo que te llevas, quiz)

**Objetivo:** correr todo el proyecto con un día nuevo en cuatro líneas, y cerrar.

| Parte | Qué haces |
|---|---|
| 1 | El pipeline en cuatro líneas |
| 2 | Automatizarlo con un Job |
| 3 | Mini quiz |
| 4 | Mapa de lo aprendido |

**Requiere:** notebook 04.

In [0]:
%run ./00_Setup

## Parte 1 · El pipeline en cuatro líneas

Llega un día nuevo y cada capa procesa **solo lo nuevo**.

In [0]:
download_days(next_day(), 1)   # RAW:    un JSON nuevo en el Volumen
run_bronze()                   # BRONZE: Auto Loader lee solo ese archivo
build_silver()                 # SILVER: MERGE con los cambios de bronze (CDF)
build_gold()                   # GOLD:   recalcula solo las claves afectadas (CDF)

In [0]:
print("RAW   :", len(raw_files()), "archivos")
print("BRONZE:", spark.table(BRONZE).count(), "filas | versión", table_version(BRONZE))
print("SILVER:", spark.table(SILVER).count(), "filas | versión", table_version(SILVER))
print("GOLD  :", spark.table(GOLD).count(), "filas | versión", table_version(GOLD))
display(spark.table(STATE))

In [0]:
# Las dos consultas clave del laboratorio
display(spark.sql(f"SELECT path, commit_time FROM cloud_files_state('{CHECKPOINT_PATH}') ORDER BY commit_time DESC LIMIT 3"))
display(spark.sql(f"SELECT _commit_version, _change_type, count(*) AS filas FROM table_changes('{SILVER}', 0) GROUP BY 1, 2 ORDER BY 1, 2"))

## Parte 2 · Automatizarlo con un Job

En el menú lateral: **Jobs & Pipelines → Create → Job**, con tres tareas en orden:
`01_RAW_API_a_Volumen` → `02_Bronze_Auto_Loader` → `07_Pipeline_Completo_y_Cierre`
(o una sola tarea con este notebook, que ya llama a todo). Programa el job cada día: Auto Loader y el CDF se encargan de
que cada corrida procese solo lo que llegó. En la Free Edition corre sobre cómputo serverless.

## Parte 3 · Mini quiz (slide 34)

1. **Delta Lake:** ¿qué aporta `_delta_log` a unos archivos Parquet?
2. **Time travel:** ¿qué pasa con el time travel después de un `VACUUM`?
3. **Change Data Feed:** ¿cuándo conviene activar CDF en una tabla?
4. **Unity Catalog:** ¿qué significa cada parte de `workspace.lab_tvmaze_m34.silver_episodes`?
5. **Auto Loader:** ¿cómo sabe qué archivos ya leyó?

*(Antes de seguir, piensa tu respuesta.)*

In [0]:
respuestas = {
    "1. Delta Lake": "Un log de transacciones: de ahí salen ACID, validación de esquema, versiones/time travel y CDF (notebooks 02 y 05).",
    "2. Time travel": "VACUUM borra los archivos de versiones viejas: después ya no se puede viajar a ellas; por defecto conserva 7 días (notebook 05).",
    "3. CDF": "Tablas grandes con pocos cambios por corrida, o cuando hay que propagar borrados: la siguiente capa procesa solo lo que cambió (notebooks 03 y 04).",
    "4. Unity Catalog": "workspace = catálogo, lab_tvmaze_m34 = esquema, silver_episodes = tabla; los permisos se heredan hacia abajo (notebook 06).",
    "5. Auto Loader": "Por el checkpoint: cloud_files_state(checkpoint) muestra cada archivo procesado; un archivo nunca entra dos veces (notebook 02).",
}
for pregunta, respuesta in respuestas.items():
    print(f"{pregunta}\n   → {respuesta}\n")

## Parte 4 · Mapa de lo aprendido

| Tema de los slides | Notebook | Evidencia |
|---|---|---|
| Delta Lake: log, ACID, versiones | 02, 05 | `DESCRIBE HISTORY`, `VERSION AS OF`, `RESTORE` |
| Crear tablas Delta, `MERGE`, esquema | 02, 03 | `CREATE TABLE ... TBLPROPERTIES`, `MERGE`, `_rescued_data` |
| OPTIMIZE / ZORDER / VACUUM, `DeltaTable` | 05 | Métricas de `OPTIMIZE`, `VACUUM DRY RUN`, `dt.history()` |
| Change Data Feed | 03, 04 | `table_changes`, `_change_type`, `pipeline_state` |
| Auto Loader | 02 | `cloudFiles`, `cloud_files_state`, re-ejecución sin duplicados |
| Unity Catalog: jerarquía, objetos, permisos, linaje | 06 | `information_schema`, vista, `GRANT`, Lineage |

## Para seguir practicando

1. **Segunda tabla gold con CDF:** `gold_show_summary` (una fila por serie: episodios, primera y última fecha, rating promedio).
2. **Esquema que cambia:** agrega `summary STRING` a `ESQUEMA_EPISODIOS`, reinicia el pipeline y compara `_rescued_data` antes y después.
3. **Más días:** sube el widget `dias` del notebook 01 a `30`, reinicia y mide cuánto tarda cada capa.

**Siguiente en el curso:** Módulo 5 · Data Pipelines (Workflows y Delta Live Tables).

## 🧹 Limpieza (opcional)
Elimina **todo** lo creado por el laboratorio. Quita el `#` para ejecutarla.

In [0]:
# spark.sql(f"DROP SCHEMA `{CATALOG}`.{SCHEMA} CASCADE")